# 1.0 ETL For World Bank's GDP

In [47]:
# System and OS related tasks
import sys
import os
import importlib
# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [48]:
import pandas as pd

import country_converter as cc

## 1.1 Get A List of Top 15 Tea and Coffee Producing Countries

First, we need to get the list of countries in the top 15 tea and coffee producing countries to subset the GDP and climate data. To do that, we will need load the top 15 tea producing countries and also top 15 coffee producing countries.

Below we prepare a variable to hold the column names.

In [49]:
# define the variables for reading in the tea and coffee files
raw_fao_dir = '../data/processed'
filename_tea = "df_fao_tea_top15_countries.csv"
filename_coffee = "df_fao_coffee_top15_countries.csv"
data_columns = [
        'Area_Code'
        , 'Area_Code_M49'
        , 'Area'
        , 'Item_Code'
        , 'Item_Code_CPC'
        , 'Item'
        , 'Element_Code'
        , 'Element'
        , 'Year_Code'
        , 'Year'
        , 'Unit'
        , 'Value'
        , 'Flag'
        , 'Note'
        , 'Country_ISO3']
column_dtypes = {
    'Area_Code': int
    , 'Area_Code_M49': str
    , 'Area' : str
    , 'Item_Code' : int
    , 'Item_Code_CPC': str
    , 'Item' : str
    , 'Element_Code' : int
    , 'Element' : str
    , 'Year_Code' : int
    , 'Year': int
    , 'Unit' : str
    , 'Value' : float
    , 'Flag' : str
    , 'Note' : str
    , 'Country_ISO3' : str
}


Get the top 15 tea producting countries. 

In [50]:
# Read in the top 15 tea countries file
df_fao_tea_top15_countries = pd.read_csv(f"{raw_fao_dir}/{filename_tea}"
                                         , header=0
                                         , names=data_columns
                                         , dtype=column_dtypes 
                                        )
df_fao_tea_top15_countries.head(3)

,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note,Country_ISO3
0,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2000,2000,ha,38620.0,A,NaN,ARG
1,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2001,2001,ha,37420.0,A,NaN,ARG
2,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2002,2002,ha,36400.0,A,NaN,ARG


In [51]:
# get a unique list of country code for tea producing countries
unique_tea_countryISO3_set = set(df_fao_tea_top15_countries["Country_ISO3"])
unique_tea_countryISO3_set 

{'ARG',
 'BGD',
 'CHN',
 'IDN',
 'IND',
 'IRN',
 'JPN',
 'KEN',
 'LKA',
 'MWI',
 'RWA',
 'TUR',
 'TZA',
 'UGA',
 'VNM'}

Get the list of top 15 coffee producing countries:

In [52]:
# Read in the top 15 coffee countries file
df_fao_coffee_top15_countries = pd.read_csv(f"{raw_fao_dir}/{filename_coffee}"
                                         , header=0
                                         , names=data_columns
                                         , dtype=column_dtypes 
                                        )
df_fao_coffee_top15_countries.head(3)

,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note,Country_ISO3
0,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2000,2000,ha,2267968.0,A,NaN,BRA
1,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2001,2001,ha,2336031.0,A,NaN,BRA
2,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2002,2002,ha,2370891.0,A,NaN,BRA


In [53]:
# get a unique list of country code for coffee producing countries
unique_coffee_countryISO3_set = set(df_fao_coffee_top15_countries["Country_ISO3"])
unique_coffee_countryISO3_set 

{'BRA',
 'CIV',
 'COL',
 'CRI',
 'ETH',
 'GIN',
 'GTM',
 'HND',
 'IDN',
 'IND',
 'MEX',
 'NIC',
 'PER',
 'UGA',
 'VNM'}

Below, we merge the top 15 tea producing countries and the top 15 coffee producing countries into 1 dataframe but mark the country as either
  * producing tea only
  * producing coffee only
  * producing both tea and coffee
for further analysis

In [54]:
# Create an empty list of dictionary 
crop_countries = []

# get tea only counry using set notation
for country in (unique_tea_countryISO3_set - unique_coffee_countryISO3_set):
    crop_countries.append({'country_ISO3': country, 'crop': 'tea only'})

# get coffee only counry using set notation
for country in (unique_coffee_countryISO3_set - unique_tea_countryISO3_set):
    crop_countries.append({'country_ISO3': country, 'crop': 'coffee only'})

# get tea and coffee counry using union notation
for country in (unique_coffee_countryISO3_set & unique_tea_countryISO3_set):
    crop_countries.append({'country_ISO3': country, 'crop': 'both'})    

df_crop_countries = pd.DataFrame(crop_countries)
df_crop_countries = df_crop_countries.sort_values("country_ISO3").reset_index(drop=True)
df_crop_countries["country_name"] = cc.convert(df_crop_countries['country_ISO3'].tolist(), to='name_short')

df_crop_countries

,country_ISO3,crop,country_name
0,ARG,tea only,Argentina
1,BGD,tea only,Bangladesh
2,BRA,coffee only,Brazil
3,CHN,tea only,China
4,CIV,coffee only,Côte d'Ivoire
5,COL,coffee only,Colombia
6,CRI,coffee only,Costa Rica
7,ETH,coffee only,Ethiopia
8,GIN,coffee only,Guinea
9,GTM,coffee only,Guatemala


In [55]:
df_crop_countries.to_csv(f'{processed_dir}/df_crop_countries.csv', index=False)
print(f"Exported: {processed_dir}/df_crop_countries.csv")

Exported: ../data/processed/df_crop_countries.csv


Prepare a list of these top countries for extacting GDP and climate data.

In [56]:
country_list = sorted(list(set(df_crop_countries["country_ISO3"])))
country_list

['ARG',
 'BGD',
 'BRA',
 'CHN',
 'CIV',
 'COL',
 'CRI',
 'ETH',
 'GIN',
 'GTM',
 'HND',
 'IDN',
 'IND',
 'IRN',
 'JPN',
 'KEN',
 'LKA',
 'MEX',
 'MWI',
 'NIC',
 'PER',
 'RWA',
 'TUR',
 'TZA',
 'UGA',
 'VNM']

## 1.2 World Bank's GDP Data (wbgapi package)

### 1.2.1 Use the wbgapi Package

In [57]:
# the wbgapi is a package for accessing World Bank's data API
# See https://pypi.org/project/wbgapi/
import wbgapi as wb

In [58]:
# take a look at the GDP data available via this package
wb.series.info(q='GDP')

id,value
EG.GDP.PUSE.KO.PP,GDP per unit of energy use (PPP $ per kg of oil equivalent)
EG.GDP.PUSE.KO.PP.KD,GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent)
EG.USE.COMM.GD.PP.KD,"Energy use (kg of oil equivalent) per $1,000 GDP (constant 2021 PPP)"
EN.GHG.CO2.RT.GDP.KD,Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)
EN.GHG.CO2.RT.GDP.PP.KD,Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)
NY.GDP.DEFL.KD.ZG,"Inflation, GDP deflator (annual %)"
NY.GDP.DEFL.KD.ZG.AD,"Inflation, GDP deflator: linked series (annual %)"
NY.GDP.DEFL.ZS,GDP deflator (base year varies by country)
NY.GDP.DEFL.ZS.AD,GDP deflator: linked series (base year varies by country)
NY.GDP.DISC.CN,Discrepancy in expenditure estimate of GDP (current LCU)


It looks like `NY.GDP.PCAP.CD` for `GDP per capita (current US$)` is a good measure that can be used to compared countries with.

📝‼️‼️: For future analysis, it might be useful to look at `GDP per capita, PPP` to see how the `Purchasing Power Parity` affect cost of farming especially for countries who relied on imports of farming equipment and fuel.

In [59]:
wb.series.info("NY.GDP.PCAP.CD")

id,value
NY.GDP.PCAP.CD,GDP per capita (current US$)
,1 elements


### 1.2.2 Get the GDP Data and Do Necessary Transformation

World Bank's economic data comes in a wide format. We will need to convert it to a long format for our forward analaysis.

In [ ]:
# Get the GDP data for 25 years for the top tea and coffee producting countries
df_wb_gdp_wide = wb.data.DataFrame(
    "NY.GDP.PCAP.CD"
    , country_list # country_ISO
    , time=range(2000,2025) # 2025 so that 2024 is included.
    , labels=True
).reset_index()

df_wb_gdp_wide.head(5)

,economy,Country,YR2000,YR2001,YR2002,YR2003,YR2004,YR2005,YR2006,YR2007,...,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,VNM,Viet Nam,404.029784,419.205678,445.132862,497.117089,565.452285,710.746760,807.756645,925.640338,...,2577.568853,2735.060438,2956.109921,3222.310031,3440.900254,3534.039535,3704.193559,4147.697772,4323.350320,4717.290287
1,UGA,Uganda,258.050339,236.021189,241.984688,250.715437,292.381858,330.405403,346.518255,401.389131,...,862.934610,752.696572,765.218332,792.165700,822.171150,845.766464,882.791718,963.067314,1002.309139,1077.912785
2,TZA,Tanzania,401.698151,396.638733,402.649048,422.183594,450.385284,483.334869,475.746735,543.198303,...,939.127319,953.010742,986.674011,1023.106262,1063.322510,1117.415283,1159.856567,1207.520630,1224.002197,1186.716919
3,TUR,Turkiye,4199.379864,3052.225671,3591.086290,4649.636681,5979.605422,7331.854294,7990.082514,9766.917345,...,11064.649450,10984.364317,10756.387768,9684.118199,9395.233780,8798.117917,9981.763984,10897.839786,13375.094728,15892.715729
4,RWA,Rwanda,251.869264,237.308846,234.043017,249.012275,269.540446,324.016864,356.995023,426.234183,...,733.998630,729.519134,758.300967,771.773511,810.051412,778.701499,829.544845,975.469324,1027.034468,999.654562


The wbgapi data is in a wide format. We need it to be in the long format so that it is in the same granularity as the FAO data.

In [61]:
#Convert the wide format data to long
## using the economy and country columns to unpivot the wide data to long data such that each long for the values of economy-country-year
df_wb_gdp_long = pd.melt(
    df_wb_gdp_wide
    , id_vars=['economy', 'Country']
    , var_name="year"
    , value_name="gdp_per_capita"
    )

df_wb_gdp_long.head(10)


,economy,Country,year,gdp_per_capita
0,VNM,Viet Nam,YR2000,404.029784
1,UGA,Uganda,YR2000,258.050339
2,TZA,Tanzania,YR2000,401.698151
3,TUR,Turkiye,YR2000,4199.379864
4,RWA,Rwanda,YR2000,251.869264
5,PER,Peru,YR2000,1945.413385
6,NIC,Nicaragua,YR2000,1017.312443
7,MWI,Malawi,YR2000,224.224159
8,MEX,Mexico,YR2000,7524.027138
9,LKA,Sri Lanka,YR2000,860.199884


In [62]:
# remove the unnecessary "YR" from the year value and convert year into integar
df_wb_gdp_long["year"] = df_wb_gdp_long["year"].str.replace("YR","")

df_wb_gdp_long["year"].astype(int)

df_wb_gdp_long.head(10)

,economy,Country,year,gdp_per_capita
0,VNM,Viet Nam,2000,404.029784
1,UGA,Uganda,2000,258.050339
2,TZA,Tanzania,2000,401.698151
3,TUR,Turkiye,2000,4199.379864
4,RWA,Rwanda,2000,251.869264
5,PER,Peru,2000,1945.413385
6,NIC,Nicaragua,2000,1017.312443
7,MWI,Malawi,2000,224.224159
8,MEX,Mexico,2000,7524.027138
9,LKA,Sri Lanka,2000,860.199884


Take a look at the economy values.

In [63]:
df_wb_gdp_long['economy'].value_counts()

economy
VNM    25
UGA    25
TZA    25
TUR    25
RWA    25
PER    25
NIC    25
MWI    25
MEX    25
LKA    25
KEN    25
JPN    25
IRN    25
IND    25
IDN    25
HND    25
GTM    25
GIN    25
ETH    25
CRI    25
COL    25
CIV    25
CHN    25
BRA    25
BGD    25
ARG    25
Name: count, dtype: int64

According to the wbgapi documentation, it is not clear that `economy` is country names based on ISO3 standard.  In fact, the documentation mentioned about it being refered to region areas. For completeness sake, we can use the country_converter package to clean the data. 

In [64]:
coco = cc.CountryConverter()

df_wb_gdp_long["country_ISO3"] = coco.pandas_convert(
    series=df_wb_gdp_long['Country'],
    to='ISO3'
)

df_wb_gdp_long["country_ISO3"].value_counts()

country_ISO3
VNM    25
UGA    25
TZA    25
TUR    25
RWA    25
PER    25
NIC    25
MWI    25
MEX    25
LKA    25
KEN    25
JPN    25
IRN    25
IND    25
IDN    25
HND    25
GTM    25
GIN    25
ETH    25
CRI    25
COL    25
CIV    25
CHN    25
BRA    25
BGD    25
ARG    25
Name: count, dtype: int64

As we can see in the World Bank's data that a country "Vietnam" is spelt as "Viet Nam" so we can add a proper ISO3 county name column as well.

In [65]:
df_wb_gdp_long["country_name"] = cc.convert(df_wb_gdp_long['country_ISO3'].tolist(), to='name_short')
df_wb_gdp_long


,economy,Country,year,gdp_per_capita,country_ISO3,country_name
0,VNM,Viet Nam,2000,404.029784,VNM,Vietnam
1,UGA,Uganda,2000,258.050339,UGA,Uganda
2,TZA,Tanzania,2000,401.698151,TZA,Tanzania
3,TUR,Turkiye,2000,4199.379864,TUR,Türkiye
4,RWA,Rwanda,2000,251.869264,RWA,Rwanda
...,...,...,...,...,...,...
645,CIV,Cote d'Ivoire,2024,2727.893522,CIV,Côte d'Ivoire
646,CHN,China,2024,13303.148154,CHN,China
647,BRA,Brazil,2024,10310.548878,BRA,Brazil
648,BGD,Bangladesh,2024,2593.416117,BGD,Bangladesh


📌 In notebook 06_Hypothesis_Testing, it was discovered that Year 2024 GDP per capita data was missing. This was because the wbgapi year range should be defined as (2000-2025) to include year 2024.

In [70]:
df_wb_gdp_long["year"].value_counts()

year
2000    26
2001    26
2002    26
2003    26
2004    26
2005    26
2006    26
2007    26
2008    26
2009    26
2010    26
2011    26
2012    26
2013    26
2014    26
2015    26
2016    26
2017    26
2018    26
2019    26
2020    26
2021    26
2022    26
2023    26
2024    26
Name: count, dtype: int64

### 1.2.3 Export World Bank's GDP data to CSV for Forward Analysis

In [68]:
df_wb_gdp_long.to_csv(f'{processed_dir}/df_wb_gdp_long.csv', index=False)
print(f"Exported: {processed_dir}/df_wb_gdp_long.csv")

Exported: ../data/processed/df_wb_gdp_long.csv
